#### Criação da tabela com schema definido

In [0]:
create table if not exists 
  workspace.pj_sales.tb_dim_car_gold (
     _sk_car bigint generated by default as identity
    ,car_brand string
    ,car_model string
    ,car_part string
    ,valid_from timestamp
    ,valid_to timestamp
  )

In [0]:
insert into workspace.pj_sales.tb_dim_car_gold
values 
   (-1, 'UNKNOWN', 'UNKNOWN', 'UNKNOWN', null, null)

#### Criação da view temporária deduplicada por data de updating, de ingestion e de sale

In [0]:
create or replace temp view vw_dim_car as(
  with dedup as (
    select
       car_brand
      ,car_model
      ,car_part
      ,row_number() over (
        partition by 
           car_brand
          ,car_model
          ,car_part
        order by 
           sale_date desc
          ,updating_timestamp desc
          ,ingestion_timestamp desc -- ! Adicionar o updating como partição
      )                                                             as order_by_date
    from workspace.pj_sales.tb_sales_silver
    --where part_quantity > 0 -- remover, alocar para os cálculos apenas
  )
  select
     car_brand
    ,car_model
    ,car_part
    ,from_utc_timestamp(current_timestamp(), 'America/Sao_Paulo')   as valid_from
    ,null                                                           as valid_to
  from dedup
  where order_by_date = 1
)

#### Sistema de atualização de dados com merge e SCD2
#####Criação de uma view temporária para fazer o merge
A view consiste na união de duas tabelas completamente iguais, com exceção da adição de uma coluna que as divide em tabela para update e tabela para insert

Assim, forma-se uma tabela com dados duplos porém com duas flags diferentes

#####Merge com a view
Na hora do merge, é verificado se o dado está válido e se o dado correspondente a ele está com a flag de atualização (update)

Quando o dado precisa ser atualizado, ou seja, é válido e o dado correspondente possui a flag de update, então o dado antigo sofre atualização em sua coluna `valid_to`, tornando-o inválido

Quando o merge não dá match, ou seja, qualquer condição pode ser negada, indicando dado novo ou, caso a condição de `src.action = 'u'` seja a **única** condição negada, indica que o dado é o mesmo, ou seja, já foi atualizado a data em seu registro original, mas agora precisa ser inserido como um dado válido, por isso é indicado com a flag `'i'`. De ambas as formas, o dado é inserido com a coluna `valid_to` nula, indicando que é o registro mais recente da dimensão

In [0]:
create or replace temp view vw_dim_car_merge as
select
  vcar.*,
  'u' as action
from vw_dim_car vcar -- Aqui o registro serve para ser atualizado
union all
select
  vcar.*,
  'i' as action
from vw_dim_car vcar -- Aqui o registro serve para ser inserido

In [0]:
merge into workspace.pj_sales.tb_dim_car_gold tg
using vw_dim_car_merge src
on  tg.car_brand = src.car_brand
and tg.car_model = src.car_model
and tg.car_part = src.car_part
and tg.valid_to is null
and src.action = 'u'
when matched and src.action = 'u' then
  update set
    tg.valid_to = src.valid_from
when not matched and src.action = 'i' then
  insert (
     car_brand
    ,car_model
    ,car_part
    ,valid_from
    ,valid_to
  )
  values (
     src.car_brand
    ,src.car_model
    ,src.car_part
    ,src.valid_from
    ,src.valid_to
  )



In [0]:
drop view vw_dim_car;
drop view vw_dim_car_merge;